In [53]:
import os
!pip install -q transformers==4.57.3
!pip install -q accelerate==1.12.0
!pip install -q bitsandbytes==0.49.1
!pip install -q langchain==1.4.1
!pip install -q langchain-community==0.4.1
!pip install -q langchain-core==1.4.1
!pip install -q langchain-text-splitters
!pip install -q pymupdf
!pip install -q langchain-huggingface
!pip install -q chromadb==1.4.1
!pip install -q huggingface-hub==0.36.0
!pip install -q rank-bm25
!pip install -q langchain-chroma

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 1.1.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 1.2.6 which is incompatible.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 1.2.6 which is incompatible.
langchain-huggingface 1.2.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 1.2.6 which is incompatible.


In [2]:
import os
import torch
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever, EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.retrievers import BM25Retriever
from langchain_core.stores import InMemoryByteStore
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [3]:
!wget -O buku_panduan_gen_ai.pdf "https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb"

--2026-05-08 11:32:03--  https://drive.google.com/uc?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.google.com (drive.google.com)... 108.177.98.139, 108.177.98.113, 108.177.98.138, ...
Connecting to drive.google.com (drive.google.com)|108.177.98.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb [following]
--2026-05-08 11:32:03--  https://drive.usercontent.google.com/download?id=1fEZDjLhh6fqiY_Bi9uKY3GSOStdgUHpb
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 108.177.98.132, 2607:f8b0:400e:c1e::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|108.177.98.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 655208 (640K) [application/octet-stream]
Saving to: ‘buku_panduan_gen_ai.pdf’

buku_panduan_gen_ai 100%[===================>] 639.85K  --.-KB/s    in 0.005s  

2026-05-08 11:32:04 (12

In [4]:
file_path = "/content/buku_panduan_gen_ai.pdf"
loader = PyMuPDFLoader(file_path)
documents = loader.load()

In [5]:
# Cek berapa halaman yang berhasil dimuat
print(f"Total halaman: {len(documents)}")

# Intip isi halaman kedua
print(f"Isi halaman 1: {documents[2].page_content[:500]}")

Total halaman: 42
Isi halaman 1: 2
Daftar Isi
BAB I: PENDAHULUAN	
5
1.1. Latar Belakang	
5
1.2. Ruang Lingkup Penggunaan	
6
1.3. Tujuan Penggunaan	
6
BAB II: PEMAHAMAN TEKNOLOGI GENERATIVE AI	
8
2.1. Perbedaan AI dan Generative AI	
8
2.2. Prinsip Kerja Generative AI	
11
2.3. Jenis-Jenis Generative AI	
11
2.4. Manfaat Generative AI Bagi Mahasiswa	
12
2.5. Apa yang Generative AI Dapat Lakukan	
12
2.6. Apa yang Generative AI Tidak Dapat Lakukan	
12
2.7. Peluang Pemanfaatan Generative AI bagi Mahasiswa	
13
2.8. Tantangan Penggunaan


In [6]:
# Parent Chunk
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
# Child Chunk
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
vectorstore = Chroma(
    collection_name="split_parents",
    embedding_function=embedding_model
)

/tmp/ipykernel_27485/3269059110.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [9]:
# Store untuk Parent
docstore = InMemoryByteStore()

In [10]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_type="similarity",
    search_kwargs={"k": 10}
)

# Masukkan Data
retriever.add_documents(documents)

In [11]:
query = "Framework apa yang dapat digunakan untuk memastikan penggunaan GenAI yang efektif?"
relevant_docs = retriever.invoke(query)

for i, doc in enumerate(relevant_docs, start=1):
    print(f"--- Dokumen {i} ---")
    print(doc.page_content)
    print()

--- Dokumen 1 ---
17
3.3. T.U.C.E. Framework (Think, Use, Check, En-
hance)
Untuk memastikan penggunaan GenAI yang efektif dan bertanggung jawab, maha-
siswa dapat mengingat Framework berikut ini:
1.	 Think (Pikirkan Sebelum Menggunakan GenAI)
a.	 Tentukan tujuan penggunaan GenAI: Apakah GenAI benar-benar diperlukan?
b.	 Pastikan GenAI digunakan sebagai alat bantu, bukan pengganti pemikiran kritis.
c.	 Periksa aturan mata kuliah atau dosen terkait penggunaan GenAI dalam tugas.
2.	 Use (Gunakan GenAI dengan Bijak)
a.	 Pilih alat GenAI yang sesuai dengan kebutuhan (ChatGPT untuk teks, DALL-E 
untuk gambar, dll.).
b.	 Masukkan instruksi yang jelas dan spesifik agar hasilnya relevan.
c.	 Hindari memasukkan data pribadi atau informasi sensitif ke dalam GenAI.
3.	 Check (Periksa Hasil dan Verifikasi Informasi)
a.	 Cek keakuratan dan kredibilitas hasil GenAI dengan sumber akademik yang valid.
b.	 Pastikan tidak ada plagiarisme dan berikan atribusi jika diperlukan.
c.	 Evaluasi apakah GenAI me

In [12]:
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 10

In [13]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever],
    weights=[0.5, 0.5]
)

In [14]:
query = "Apa sih itu GenAI detector"
hybrid_relevant_docs = hybrid_retriever.invoke(query)

for i, doc in enumerate(hybrid_relevant_docs, start=1):
    print(f"--- Dokumen {i} ---")
    print(doc.page_content)
    print()

--- Dokumen 1 ---
31
BAB V: Tanya Jawab Seputar 
Penggunaan Generative AI Dalam Akademik
Bagian ini berisi pertanyaan umum yang sering diajukan mahasiswa terkait penggu-
naan GenAI dalam lingkungan akademik, serta batasan dan aturan spesifik yang ber-
laku.
1.	 Kapan mahasiswa boleh menggunakan GenAI dalam tugas akademik? 
Mahasiswa diperbolehkan menggunakan GenAI dalam tugas akademik jika:
1.	 GenAI digunakan sebagai alat bantu dalam brainstorming, merangkum, atau 
menyusun ide.
2.	 GenAI membantu mengoreksi tata bahasa dan meningkatkan kualitas tulisan.
3.	 Mahasiswa tetap memahami dan menyesuaikan hasil GenAI dengan analisis dan 
pemikiran mereka sendiri.
4.	 Sumber GenAI dicantumkan dengan format kutipan yang sesuai.
2.	 Kapan GenAI tidak boleh digunakan dalam tugas akademik?
Mahasiswa tidak diperbolehkan menggunakan GenAI dalam tugas akademik jika:
1.	 Dosen secara eksplisit melarang penggunaan GenAI dalam tugas tertentu.
2.	 Mahasiswa mengandalkan GenAI sepenuhnya tanpa memahami 

In [37]:
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")

compressor = CrossEncoderReranker(model=model, top_n=5)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever,
)

In [38]:
final_docs = compression_retriever.invoke(query)


from IPython.display import display, Markdown

# Pastikan menggunakan 'final_docs' hasil dari compression_retriever
print(f"✅ Berhasil menyaring {len(final_docs)} dokumen terbaik dengan Reranker:\n")

for i, doc in enumerate(final_docs, start=1):
    # Ambil nomor halaman dari metadata
    page_num = doc.metadata.get('page', 'N/A')

    # Visualisasi rapi
    display(Markdown(f"""
---
### 📄 Dokumen {i} (Halaman: {page_num})
> **Isi Konten:**
{doc.page_content}

---
"""))

✅ Berhasil menyaring 5 dokumen terbaik dengan Reranker:




---
### 📄 Dokumen 1 (Halaman: 25)
> **Isi Konten:**
25
BAB IV: Ruang Lingkup Integritas Akademik 
Dalam Penggunaan Generative AI
Penggunaan Generative AI (GenAI) dalam lingkungan akademik harus tetap menjun-
jung tinggi integritas akademik, yaitu prinsip kejujuran, tanggung jawab, dan etika da-
lam belajar dan berkarya. Untuk memastikan hal ini, diperlukan strategi yang mencak-
up pencegahan, pembinaan, dan penanggulangan terhadap penyalahgunaan AI dalam 
tugas akademik.
4.1. Pencegahan: Mencegah Penyalahgunaan Gen-
erative AI Sejak Awal
4.1.1. Memahami Kemampuan Dosen dalam Mengetahui Penggunaan Genera-
tive AI pada Mahasiswa
Mahasiswa perlu menyadari bahwa dosen memiliki berbagai cara untuk mendeteksi 
penggunaan GenAI dalam tugas akademik. Beberapa metode yang digunakan meliputi:
1.	 Penggunaan GenAI Detector: Dosen dapat menggunakan alat pendeteksi GenAI 
seperti Turnitin AI Detector, GPTZero, dan Originality.ai untuk mengidentifikasi 
teks yang dihasilkan oleh GenAI.
2.	 Analisis Gaya Tulisan: Dosen dapat membandingkan tugas yang dikumpulkan 
dengan gaya tulisan mahasiswa sebelumnya untuk mendeteksi perbedaan signifikan.
3.	 Validasi Sumber dan Referensi: GenAI sering kali menghasilkan referensi pal-
su atau tidak valid. Dosen dapat memeriksa apakah referensi yang digunakan be-
nar-benar ada dan relevan.
4.	 Tes Lisan atau Diskusi: Dosen dapat menguji pemahaman mahasiswa terhadap 
tugas yang mereka serahkan melalui wawancara atau diskusi untuk memastikan 
mereka benar-benar memahami isi tugas tersebut.
5.	 Pola Jawaban yang Terlalu Umum atau Tidak Spesifik: Hasil yang dihasilkan 
oleh GenAI cenderung bersifat umum dan tidak menunjukkan pemahaman men-
dalam. Dosen dapat mengidentifikasi tugas yang memiliki jawaban terlalu generik 
atau tidak memiliki analisis kritis. 

---



---
### 📄 Dokumen 2 (Halaman: 31)
> **Isi Konten:**
31
BAB V: Tanya Jawab Seputar 
Penggunaan Generative AI Dalam Akademik
Bagian ini berisi pertanyaan umum yang sering diajukan mahasiswa terkait penggu-
naan GenAI dalam lingkungan akademik, serta batasan dan aturan spesifik yang ber-
laku.
1.	 Kapan mahasiswa boleh menggunakan GenAI dalam tugas akademik? 
Mahasiswa diperbolehkan menggunakan GenAI dalam tugas akademik jika:
1.	 GenAI digunakan sebagai alat bantu dalam brainstorming, merangkum, atau 
menyusun ide.
2.	 GenAI membantu mengoreksi tata bahasa dan meningkatkan kualitas tulisan.
3.	 Mahasiswa tetap memahami dan menyesuaikan hasil GenAI dengan analisis dan 
pemikiran mereka sendiri.
4.	 Sumber GenAI dicantumkan dengan format kutipan yang sesuai.
2.	 Kapan GenAI tidak boleh digunakan dalam tugas akademik?
Mahasiswa tidak diperbolehkan menggunakan GenAI dalam tugas akademik jika:
1.	 Dosen secara eksplisit melarang penggunaan GenAI dalam tugas tertentu.
2.	 Mahasiswa mengandalkan GenAI sepenuhnya tanpa memahami isi tugas yang 
dihasilkan.
3.	 GenAI digunakan untuk menyelesaikan tugas ujian atau tugas individu yang 
mengharuskan pemikiran mandiri.
4.	 Hasil dari GenAI disalin tanpa penyuntingan atau atribusi yang jelas.
5.	 Menggunakan GenAI untuk menghasilkan data penelitian tanpa eksperimen yang 
valid.
6.	 Menggunakan GenAI untuk menjawab soal ujian atau tugas.
3.	 Bagaimana dosen dapat mengetahui jika mahasiswa menggunakan GenAI?
Dosen dapat menggunakan berbagai alat deteksi GenAI seperti Turnitin AI Detector, 
GPTZero, atau Originality.ai untuk mengevaluasi tugas yang dikumpulkan. Selain 
itu, dosen juga dapat menganalisis gaya tulisan, memeriksa validitas referensi, serta 
melakukan diskusi atau tes lisan untuk memastikan pemahaman mahasiswa.
4.	 Di mana mahasiswa dapat mengetahui aplikasi-aplikasi GenAI yang dapat 
mereka gunakan? 

---



---
### 📄 Dokumen 3 (Halaman: 8)
> **Isi Konten:**
8
BAB II: Pemahaman Teknologi Generative AI
2.1. Perbedaan AI dan Generative AI
Sebelum memahami cara kerja Generative AI (GenAI), penting bagi mahasiswa untuk 
memahami perbedaan antara Artificial Intelligence (AI) secara umum dan GenAI se-
cara khusus.
1.	 Artificial Intelligence (AI)
•	 AI adalah teknologi yang dirancang untuk meniru kecerdasan manusia dalam berb-
agai tugas seperti analisis data, pengenalan pola, dan pengambilan keputusan.
•	 Contoh AI meliputi asisten virtual seperti Siri dan Google Assistant, algoritma pen-
carian Google, dan sistem rekomendasi Netflix atau Spotify.
•	 AI umumnya bersifat berbasis aturan atau pembelajaran mesin yang membantu ma-
nusia dalam mengambil keputusan.
2. Generative AI (GenAI)
•	 GenAI adalah cabang dari AI yang mampu menghasilkan konten baru, seperti teks, 
gambar, video, dan suara, berdasarkan data yang telah dipelajarinya.
•	 Model GenAI menggunakan teknik deep learning dan neural networks untuk men-
ciptakan output yang menyerupai karya manusia.
•	 Contoh GenAI meliputi ChatGPT (pembuatan teks), DALL-E (pembuatan gam-
bar), dan Sora (pembuatan video).
Berikut adalah perbedaan utama antara AI (Artificial Intelligence) dan Generative 
AI (GenAI): 

---



---
### 📄 Dokumen 4 (Halaman: 33)
> **Isi Konten:**
33
6.	 Seperti apakah contoh penyalahgunaan GenAI?
A.	Penyalahgunaan dalam Tugas Esai atau Makalah:
Kasus: Mahasiswa menyalin seluruh isi makalah dari GenAI tanpa melakukan mod-
ifikasi atau analisis tambahan.
Mengapa Salah?
1.	 Tidak ada kontribusi pemikiran mahasiswa dalam tugas.
2.	 GenAI digunakan untuk membuat seluruh isi tugas tanpa pemahaman dari ma-
hasiswa.
3.	 Tidak mencantumkan bahwa GenAI digunakan dalam tugas.
Cara yang Benar:
1.	 Menggunakan GenAI untuk mencari referensi atau membuat kerangka esai, teta-
pi isi utama tetap dikembangkan sendiri.
2.	 Jika GenAI digunakan, mahasiswa harus menyebutkan peran GenAI dalam ba-
gian atribusi tugas.
Contoh Penyalahgunaan:
“Perubahan iklim adalah fenomena global yang disebabkan oleh aktivitas manu-
sia...” (Seluruh esai dihasilkan oleh AI tanpa perubahan dari mahasiswa).
Cara yang Benar:
“Dalam makalah ini, ChatGPT digunakan untuk menyusun kerangka awal dan mer-
angkum teori perubahan iklim. Namun, seluruh analisis dan kesimpulan disusun 
berdasarkan penelitian mandiri.”
B.	Penyalahgunaan dalam Proposal Penelitian atau Skripsi
Kasus: Mahasiswa menggunakan GenAI untuk menghasilkan daftar pustaka den-
gan referensi palsu yang tidak pernah diterbitkan (AI hallucination).
Mengapa Salah?
1.	 Daftar pustaka yang dibuat GenAI sering kali berisi referensi yang tidak dapat 
diverifikasi.
2.	 Mencantumkan sumber yang tidak ada merupakan bentuk fabrikasi akademik.
3.	 Dosen atau penguji tidak dapat menelusuri sumber yang digunakan. 

---



---
### 📄 Dokumen 5 (Halaman: 22)
> **Isi Konten:**
22
3.	 Apabila ingin membuat desain grafis atau ilustrasi untuk presentasi atau tu-
gas → Maka dapat menggunakan DALL-E atau Midjourney untuk menghasilkan 
gambar yang sesuai dengan kebutuhan.
4.	 Apabila membutuhkan pembuatan video edukatif atau presentasi berbasis 
GenAI → Maka dapat menggunakan Synthesia atau Sora untuk membuat video 
berbasis teks atau skrip yang disediakan.
5.	 Apabila ingin menerjemahkan teks akademik atau memperbaiki tata bahasa 
dalam tulisan → Maka dapat menggunakan DeepL atau Grammarly untuk mem-
bantu dalam penerjemahan dan proofreading.
6.	 Apabila membutuhkan bantuan dalam memahami konsep atau penjelasan 
materi sulit → Maka dapat menggunakan ChatGPT atau Khan Academy AI un-
tuk mendapatkan penjelasan interaktif dan mudah dipahami.
7.	 Apabila ingin mengelola waktu belajar dan mengatur tugas akademik dengan 
lebih baik → Maka dapat menggunakan Notion AI, Todoist, atau Microsoft To 
Do untuk membuat jadwal dan daftar tugas.
3.7. Contoh Kasus Studi Implementasi Generative 
AI pada Mahasiswa
Sering mengalami writer’s block saat menulis tugas? Atau ingin presentasi Anda leb-
ih menarik? GenAI bisa membantu Anda mendapatkan inspirasi! GenAI bisa menja-
di ‘rekan brainstorming’ yang memberikan ide-ide segar, merancang desain menarik, 
hingga menyusun skrip video pembelajaran. Manfaatkan GenAI untuk membuat pros-
es belajar lebih seru dan inovatif!
Untuk membayangkan bagaimana GenAI dapat diterapkan dalam lingkungan akade-
mik, berikut beberapa kasus studi yang menggambarkan penggunaannya dalam berb-
agai skenario: 

---


In [39]:
# Konfigurasi Kuantisasi 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "unsloth/Llama-3.2-3B-Instruct"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [40]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=1000,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Device set to use cuda:0


In [41]:
template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Anda adalah asisten AI yang bertugas membantu pengguna.
Gunakan hanya informasi yang tersedia pada konteks berikut untuk menjawab pertanyaan.
Jika jawaban tidak ditemukan dalam konteks tersebut, sampaikan dengan jujur bahwa Anda tidak mengetahui jawabannya dan jangan membuat asumsi atau jawaban tambahan.
Berikan jawaban secara singkat dan jelas.

Context:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>

{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [42]:
rag_chain = (
    {"context": compression_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [43]:
def ask_question(query):
    print(f"🔍 Pertanyaan: {query}")
    print("-" * 50)

    # 1. Jalankan Chain untuk mendapatkan jawaban dari LLM
    response = rag_chain.invoke(query)

    print("🤖 Jawaban:")
    print(response)
    print("-" * 50)

    # 2. Tampilkan sumber referensi (Gunakan compression_retriever agar sinkron dengan jawaban)
    # Ini adalah dokumen yang benar-benar dibaca oleh Llama-3.2 tadi
    docs = compression_retriever.invoke(query)

    print("📚 Sumber Referensi (Terverifikasi Reranker):")
    # Kita gunakan set() agar jika ada potongan dari halaman yang sama tidak muncul double
    pages = sorted(list(set([doc.metadata.get('page', '?') for doc in docs])))

    for i, page in enumerate(pages, start=1):
        print(f"{i}. PDF Halaman: {page}")



In [47]:
# --- CARA PAKAI ---
ask_question("Apa itu framework T.U.C.E?")

🔍 Pertanyaan: Apa itu framework T.U.C.E?
--------------------------------------------------
🤖 Jawaban:
Framework T.U.C.E. adalah sebuah model yang digunakan untuk memastikan penggunaan Generative AI (GenAI) dengan bijak dan bertanggung jawab dalam lingkungan akademik. Acronim T.U.C.E. merupakan singkatan dari empat langkah utama:

T - Think (Pikirkan)
U - Use (Gunakan)
C - Check (Periksa)
E - Enhance (Tingkatkan)

Framework ini dirancang untuk membantu mahasiswa menggunakan GenAI dengan efektif dan bertanggung jawab, serta mencegah penyalahgunaan teknologi ini dalam konteks akademik.
--------------------------------------------------
📚 Sumber Referensi (Terverifikasi Reranker):
1. PDF Halaman: 2
2. PDF Halaman: 17
3. PDF Halaman: 34
4. PDF Halaman: 38


In [46]:
import torch
import gc

# Hapus variabel yang memakan banyak tempat jika ada
if 'model' in locals():
    del model
if 'tokenizer' in locals():
    del tokenizer

# Paksa Garbage Collection dan kosongkan cache CUDA
gc.collect()
torch.cuda.empty_cache()